# Models - finding a better way

This version is much like the Models notebook only this one is more to to standard.

In [1]:
'''
This isn't strictly needed. However it solves this annoying pandas error:


/opt/homebrew/Caskroom/miniforge/base/envs/supervised/lib/python3.9/site-packages/xgboost/compat.py:36: FutureWarning: pandas.Int64Index is deprecated and will be removed from pandas in a future version. Use pandas.Index with the appropriate dtype instead.
  from pandas import MultiIndex, Int64Index
  
The problem is solved with xgboost 1.6 but I don't want to use pip in this case and the conda package is currently 1.5.1  
'''

import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


from sklearn.pipeline import Pipeline, make_pipeline
from sklearn import preprocessing
from sklearn import set_config

from sklearn.preprocessing import StandardScaler, OneHotEncoder, PolynomialFeatures
from sklearn.preprocessing import QuantileTransformer
from sklearn.preprocessing import OrdinalEncoder
from sklearn.preprocessing import MinMaxScaler

from sklearn.compose import TransformedTargetRegressor
from sklearn.compose import ColumnTransformer
from sklearn.compose import make_column_transformer

from sklearn.svm import SVC
from sklearn.decomposition import PCA

from sklearn.linear_model import LogisticRegression
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
import xgboost as xgb

from sklearn.feature_selection import SelectPercentile, chi2

from sklearn.metrics import r2_score, mean_squared_error,accuracy_score, make_scorer

from sklearn.model_selection import train_test_split
from sklearn.model_selection import RandomizedSearchCV
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.model_selection import GridSearchCV
from sklearn.linear_model import LogisticRegression

In [2]:
df = pd.read_pickle("../data/diamonds.pkl")

In [3]:
numerical_columns = df.select_dtypes(include=['int64', 'float64']).columns.to_list()

In [4]:
ordinal_columns = ['clarity', 'culet_size', 'cut_quality', 'polish', 'symmetry']

clarity_ord = ['F', 'IF', 'VVS1', 'VVS2', 'VS1', 'VS2', 'SI1',  'SI2', 'SI3', 'I1', 'I2','I3']
culet_size_ord = ['N', 'VS', 'S', 'M', 'SL', 'L', 'VL', 'EL', 'unknown']
cut_quality_ord = ['Ideal', 'Excellent',  'Very Good', 'Good', 'Fair', 'None', 'unknown']
polish_ord = ['Excellent','Very Good', 'Good', 'Fair', 'Poor']
symmetry_ord = ['Excellent','Very Good', 'Good', 'Fair', 'Poor']

In [5]:
nominal_columns = ['cut', 'color','lab','eye_clean','culet_condition','girdle_max', 'girdle_min', 'fancy_color_intensity','fancy_color_dominant_color','fancy_color_secondary_color','fancy_color_overtone','fluor_color', 'fluor_intensity']

In [6]:
ordinal_transformer = Pipeline(steps=[
    ('encoder', OrdinalEncoder(categories=[
        ['F', 'IF', 'VVS1', 'VVS2', 'VS1', 'VS2', 'SI1', 'SI2', 'SI3', 'I1', 'I2', 'I3'], # clarity_ord
        ['N', 'VS', 'S', 'M', 'SL', 'L', 'VL', 'EL', 'unknown'], # culet_size_ord
        ['Ideal', 'Excellent', 'Very Good', 'Good', 'Fair', 'None', 'unknown'], # cut_quality_ord
        ['Excellent', 'Very Good', 'Good', 'Fair', 'Poor'], # polish_ord
        ['Excellent', 'Very Good', 'Good', 'Fair', 'Poor'] # symmetry_ord
    ]))
])

In [7]:
nominal_transformer = Pipeline(steps=[
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])

In [8]:
# define the preprocessor with the transformers
preprocessor = ColumnTransformer(transformers=[
    ('ord', ordinal_transformer, ['clarity', 'culet_size', 'cut_quality', 'polish', 'symmetry']),
    ('nom', nominal_transformer, nominal_columns),
], remainder='passthrough', n_jobs=-1)


# define the model pipeline
model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression())
])

In [9]:
# define a scorer for cross-validation
scorer = make_scorer(accuracy_score)

In [10]:
# load the dataset and split into training and test sets

df = pd.read_pickle("../data/diamonds.pkl")
X = df.drop('total_sales_price', axis=1)
y = df['total_sales_price']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

X_train.shape, y_train.shape, X_test.shape, y_test.shape

((175762, 24), (175762,), (43941, 24), (43941,))

In [11]:
unique, counts = np.unique(y, return_counts=True)
class_counts = dict(zip(unique, counts))
least_populated_class = min(class_counts, key=class_counts.get)
print(f"Least populated class: {least_populated_class}")

#prints: "Least populated class: 202"

Least populated class: 202


### The above cell shows a least populated cell of 202, however, the following cell generates the following error messages:

##### With StratifiedKFold

UserWarning: The least populated class in y has only 1 members, which is less than n_splits=3.

##### With StratifiedShuffleSplit
ValueError: The least populated class in y has only 1 member, which is too few. The minimum number of groups for any class cannot be less than 2.

In [17]:
# cv = StratifiedKFold(n_splits=3, shuffle=True)
# scores = cross_val_score(model, X_train, y_train, cv=cv, scoring=scorer)

cv = StratifiedShuffleSplit(n_splits=3, test_size=0.3, random_state=42)
scores = cross_val_score(model, X_train, y_train, cv=cv, scoring=scorer)


ValueError: The least populated class in y has only 1 member, which is too few. The minimum number of groups for any class cannot be less than 2.

In [ ]:
# print the mean and standard deviation of the cross-validation scores
print("Accuracy: %0.2f (+/- %0.2f)" % (scores.mean(), scores.std() * 2))
